In [5]:
import sys
from pathlib import Path
from langchain_core.documents import Document
from dotenv import load_dotenv
load_dotenv()
root = Path().resolve().parent  # adjust level as needed
sys.path.insert(0, str(root))

In [6]:
import pickle
with open(root / "datasets" / "hotpotqa.pkl", "rb") as f:
    all_documents = pickle.load(f) 

In [7]:
import json
with open(root / "datasets" / "hotpotqa_eval.json", "r") as f:
    eval_dataset = json.load(f)


In [8]:
from langchain_upstage import UpstageEmbeddings
from reranker.rrf import ReciprocalRankFusion
from langchain_community.retrievers import BM25Retriever
embeddings = UpstageEmbeddings(model="embedding-passage")
bm25_retriever = BM25Retriever.from_documents(all_documents)
bm25_retriever.k = 100
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
        root / "faiss_index", 
        embeddings,
        "hotpotqa_100",
        allow_dangerous_deserialization=True  # needed in newer versions
    )
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 100})
def retrieve_document(question: str) -> list[str]:
    retrieved_docs_faiss = faiss_retriever.invoke(question)
    retrieved_docs_bm25 = bm25_retriever.invoke(question)
    retrieved_docs_faiss = ReciprocalRankFusion.calculate_rank_score(retrieved_docs_faiss)
    retrieved_docs_bm25 = ReciprocalRankFusion.calculate_rank_score(retrieved_docs_bm25)
    retrieved_docs = retrieved_docs_faiss + retrieved_docs_bm25
    rrf_docs = ReciprocalRankFusion.get_rrf_docs(retrieved_docs, cutoff=100)
    return rrf_docs

In [ ]:
import cohere
import os
from tqdm import tqdm

co = cohere.Client(api_key=os.getenv("COHERE_API_KEY"))

def rerank_document(eval_dataset: list[dict], retrieved_docs: list[list[Document]], top_n: int = 20) -> list[list[Document]]:
    rerank_results = []
    for i, entry in enumerate(tqdm(eval_dataset, desc="Reranking documents")):
        query = entry["query"]
        documents = [f"{doc.page_content}" for doc in retrieved_docs[i]]
        
        if not documents:
            rerank_results.append([])
            continue
            
        rerank_response = co.rerank(
            model="rerank-v4.0-fast", query=query, documents=documents, top_n=top_n
        )
        rerank_results.append(rerank_response.results)

    reranked_chunks_arr = []
    for idx, retrieved_doc in enumerate(retrieved_docs):
        reranked_chunks = []
        if idx < len(rerank_results):
            for result in rerank_results[idx]:
                reranked_chunks.append(retrieved_doc[result.index])
        reranked_chunks_arr.append(reranked_chunks)

    return reranked_chunks_arr

In [10]:
def recall(eval_dataset: list[dict], retrieved_docs: list[str]) -> dict:
    true_positives = 0
    false_negatives = 0

    for i, data in enumerate(eval_dataset):  
        # 중복 페이지 제거
        # reference_page_number = list({int(page) for page in row["page_number"].strip("[]").split(",")})
        reference_chunk_id = data["golden_chunk_ids"]
        retrieved_chunk_id = [doc.metadata["chunk_id"] for doc in retrieved_docs[i]]

        for chunk_id in reference_chunk_id:
            if chunk_id in retrieved_chunk_id:
                true_positives += 1
            else:
                print("index: ", i)
                print(data["query"])
                print(data["golden_chunk_ids"])
                print(retrieved_chunk_id)
                false_negatives += 1

    print(f"True Positives: {true_positives}, False Negatives: {false_negatives}")

    recall = true_positives / (true_positives + false_negatives)
    return {"recall": recall}

In [11]:
retrieved_docs = [retrieve_document(eval_dataset[i]["query"]) for i in range(len(eval_dataset))]

In [12]:
reranked_docs = rerank_document(eval_dataset, retrieved_docs, 20)

Reranking documents: 100%|██████████| 100/100 [00:32<00:00,  3.11it/s]


In [13]:
recall(eval_dataset, retrieved_docs)

index:  41
Ernest Davies' successor was born on what date?
['doc_42_chunk_1', 'doc_42_chunk_5']
['doc_42_chunk_5', 'doc_42_chunk_6', 'doc_42_chunk_10', 'doc_42_chunk_3', 'doc_42_chunk_8', 'doc_42_chunk_9', 'doc_42_chunk_7', 'doc_42_chunk_4', 'doc_37_chunk_3', 'doc_52_chunk_4', 'doc_41_chunk_4', 'doc_90_chunk_7', 'doc_70_chunk_10', 'doc_85_chunk_6', 'doc_11_chunk_4', 'doc_43_chunk_4', 'doc_31_chunk_2', 'doc_42_chunk_2', 'doc_91_chunk_3', 'doc_26_chunk_6', 'doc_78_chunk_6', 'doc_43_chunk_3', 'doc_2_chunk_10', 'doc_53_chunk_8', 'doc_68_chunk_10', 'doc_27_chunk_7', 'doc_80_chunk_1', 'doc_8_chunk_1', 'doc_90_chunk_3', 'doc_91_chunk_2', 'doc_14_chunk_9', 'doc_37_chunk_4', 'doc_12_chunk_8', 'doc_90_chunk_10', 'doc_73_chunk_4', 'doc_22_chunk_5', 'doc_84_chunk_4', 'doc_55_chunk_7', 'doc_48_chunk_6', 'doc_68_chunk_3', 'doc_52_chunk_9', 'doc_18_chunk_6', 'doc_90_chunk_4', 'doc_55_chunk_3', 'doc_22_chunk_4', 'doc_47_chunk_5', 'doc_78_chunk_7', 'doc_39_chunk_6', 'doc_62_chunk_7', 'doc_46_chunk_1', 

{'recall': 0.995}

In [14]:
recall(eval_dataset, reranked_docs)

index:  41
Ernest Davies' successor was born on what date?
['doc_42_chunk_1', 'doc_42_chunk_5']
['doc_42_chunk_5', 'doc_42_chunk_9', 'doc_42_chunk_10', 'doc_42_chunk_7', 'doc_42_chunk_6', 'doc_42_chunk_8', 'doc_90_chunk_7', 'doc_47_chunk_5', 'doc_42_chunk_3', 'doc_37_chunk_2', 'doc_12_chunk_10', 'doc_55_chunk_3', 'doc_55_chunk_2', 'doc_15_chunk_1', 'doc_42_chunk_2', 'doc_22_chunk_5', 'doc_52_chunk_4', 'doc_2_chunk_10', 'doc_73_chunk_4', 'doc_55_chunk_7']
True Positives: 199, False Negatives: 1


{'recall': 0.995}